In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.features.feature_engineer.min_features import _detect_star_players
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.utils.helpers import *
from src.utils.dataScraper import *
from live import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### Get updated lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
{}

Out Players:
{'MIN': ['Ayo Dosunmu'], 'SAS': ['Carter Bryant']}
Successfully updated C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\src\utils\team_info.py
Updated 4 teams with confirmed lineups
Updated 0 teams with questionable players


### Dataset

In [3]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
p25 = pd.read_csv('data/raw/playoff_stats/P25.csv').sort_values(by='GAME_DATE')
s25 = pd.concat([s25, p25])
s25 = _detect_star_players(s25)

s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
p26 = pd.read_csv('data/raw/playoff_stats/P26.csv').sort_values(by='GAME_DATE')
s26 = pd.concat([s26, p26])
s26 = _detect_star_players(s26)

base_df = pd.concat([s25, s26])
base_df.tail()

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,START_POSITION,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,IS_PLAYOFF,POS,AGE,TEAM_SPREAD_ODDS,GAME_TOTAL_ODDS,TEAM_SPREAD,GAME_TOTAL,STARTING,PTS_PER_MIN,AST_PER_MIN,REB_PER_MIN,IS_HOME,POSITION_ENCODED,TOP_PLAYER,SECOND_TOP_PLAYER,THIRD_TOP_PLAYER,IS_TOP_STAR,IS_TOP_1_STAR,ACTIVE_STARS_COUNT,TOP_STAR_ACTIVE,TOP_PLAYER_ACTIVE,SECOND_PLAYER_ACTIVE,THIRD_PLAYER_ACTIVE,name
27713,NaN,NaN,1038,2025-26,1630567,Scottie Barnes,Scottie,1610612761,TOR,Toronto Raptors,42500137,2026-05-03,TOR @ CLE,L,37.440000,8,14,0.571,1,1,1.000,7,7,1.00,1,8,9,6,3,0,1,0,5,5,24,0,43.8,0,0,42.0,1,37:26,1,104.6,113.2,113.2,110.6,113.2,113.2,-6.0,0.0,0.0,0.250,2.0,24.0,0.027,0.200,0.117,12.0,11.5,0.607,0.703,0.226,0.233,102.56,97.44,81.20,97.44,0.186,76,8.0,14.0,F,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.000,0.0,0.0,0.000,38,87,0.437,8,28,0.286,18,26,0.692,7,26,33,25,14.0,10,4,5,28,23,102,-12.0,96.7,104.1,116.0,116.3,-19.3,-12.2,0.658,1.79,18.7,0.226,0.547,0.387,0.143,0.483,0.518,101.9,98.0,81.67,98,0.409,1610612739,CLE,Cleveland Cavaliers,38,85,0.447,11,39,0.282,27,37,0.730,20,40,60,23,17.0,8,5,4,23,28,114,12.0,116.0,116.3,96.7,104.1,19.3,12.2,0.605,1.35,15.9,0.453,0.774,0.613,0.173,0.512,0.563,101.9,98.0,81.67,98,0.591,1,PF,24.0,NaN,NaN,8.5,209.0,1,0.641026,0.160256,0.240385,0,1,RJ Barrett,Scottie Barnes,Sandro Mamukelashvili,1,0,2,1,1,1,0,Scottie Barnes
27714,NaN,NaN,1039,2025-26,1630595,Cade Cunningham,Cade,1610612765,DET,Detroit Pistons,42500107,2026-05-03,DET vs. ORL,W,39.483333,10,18,0.556,4,6,0.667,8,10,0.80,0,1,1,12,4,0,2,2,4,10,32,29,53.2,1,0,53.0,1,39:29,1,132.1,132.9,132.9,92.4,94.7,94.7,39.7,38.2,38.2,0.462,3.0,30.8,0.000,0.025,0.013,10.3,10.4,0.667,0.714,0.300,0.305,93.85,92.39,76.99,92.39,0.177,76,10.0,18.0,G,4.07,2.86,2.0,4.0,6.0,101.0,1.0,0.0,69.0,4.0,6.0,0.667,6.0,12.0,0.500,1.0,3.0,0.333,41,80,0.513,16,33,0.485,18,22,0.818,11,30,41,30,14.0,9,6,7,21,21,116,22.0,125.2,126.1,98.7,102.2,26.4,23.9,0.732,2.14,21.9,0.366,0.688,0.539,0.152,0.613,0.647,93.9,92.0,76.67,92,0.636,1610612753,ORL,Orlando Magic,31,75,0.413,10,30,0.333,22,30,0.733,9,24,33,18,16.0,11,7,6,21,21,94,-22.0,98.7,102.2,125.2,126.1,-26.4,-23.9,0.581,1.13,14.4,0.313,0.634,0.461,0.174,0.480,0.533,93.9,92.0,76.67,92,0.364,1,PG,24.0,NaN,NaN,-8.5,201.0,1,0.810469,0.303926,0.025327,1,2,Jalen Duren,Cade Cunningham,Paul Re

### Load latest odds on file

In [4]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_odds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_odds = pd.json_normalize(data)

print("Loaded:", file.name)
team_odds.head()

Loaded: NBA_20260503_112034.json


,home_team,away_team,commence_time,bookmakers
0,Detroit Pistons,Orlando Magic,2026-05-03 19:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
1,Cleveland Cavaliers,Toronto Raptors,2026-05-03 23:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
2,New York Knicks,Philadelphia 76ers,2026-05-05 00:10:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
3,San Antonio Spurs,Minnesota Timberwolves,2026-05-05 01:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
4,Oklahoma City Thunder,Los Angeles Lakers,2026-05-06 01:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."


In [5]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = s26
ast_df = s26
reb_df = s26
min_df = s26


#load dfs lines
lines_dfs = pd.read_csv(dfs_file)
# lines_dfs = lines_dfs[lines_dfs['COMMENCE_TIME'] == current_date]
lines_dfs_pts = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['CATEGORY'] == 'player_rebounds')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
# lines_us = lines_us[lines_us['COMMENCE_TIME'] == current_date]
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]
print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")

lines_dfs_pts.head()

DFS latest pull: 2026-05-04 17:33:14
US latest pull: 2026-05-04 17:34:17


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,PrizePicks,player_points,Jalen Brunson,Over,30.5,-137,2026-05-05,2026-05-05T00:32:19Z,2026-05-04 17:33:14
1,PrizePicks,player_points,Jalen Brunson,Under,30.5,-137,2026-05-05,2026-05-05T00:32:19Z,2026-05-04 17:33:14
2,PrizePicks,player_points,Karl-Anthony Towns,Over,16.5,-137,2026-05-05,2026-05-05T00:32:19Z,2026-05-04 17:33:14
3,PrizePicks,player_points,Karl-Anthony Towns,Under,16.5,-137,2026-05-05,2026-05-05T00:32:19Z,2026-05-04 17:33:14
4,PrizePicks,player_points,OG Anunoby,Over,14.5,-137,2026-05-05,2026-05-05T00:32:19Z,2026-05-04 17:33:14


### Load my models

In [6]:
import joblib

#minutes
min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb_2026-01-16.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

#points per minute
ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb_2026-01-01.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]

#assists per minute
apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb_2026-01-01.joblib")
apm_quantile_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]

#rebounds per minute
rpm_bundle = joblib.load("src/models/saved_models/rpm_quantile_xgb_2026-01-01.joblib")
rpm_quantile_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]

### Get Min predictions and Stat Per Min predictions 

In [7]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
)
reb_preds = predict_min_times_rate(
    reb_names, min_df, reb_df, current_date,
    rate_pipeline=rpm_pipeline,
    rate_quantile_models=rpm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="REB",
)
pts_preds.head(10)

Team odds: NBA_20260503_112034.json
[SKIP] Kelly Oubre Jr: min_pipeline returned None (need >= 10 games)
[SKIP] Kelly Oubre Jr: min_pipeline returned None (need >= 10 games)


,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90,RATE_HISTORY
0,Jalen Brunson,PTS,29.21,36.74,41.57,0.3900,0.6253,0.9522,11.39,22.97,39.59,"[0.8112874779541446, 0.8272296423955192, 0.653..."
1,Karl-Anthony Towns,PTS,24.46,31.75,37.01,0.3307,0.5541,0.8732,8.09,17.60,32.31,"[1.014964216005205, 0.8611851548082837, 0.5916..."
2,OG Anunoby,PTS,27.57,35.38,40.82,0.2477,0.4985,0.7669,6.83,17.63,31.31,"[0.5298570227081582, 0.4347826086956521, 0.304..."
3,Mikal Bridges,PTS,21.43,29.72,36.57,0.1624,0.4285,0.7295,3.48,12.73,26.68,"[0.4375, 0.4381907590437703, 0.207581673503682..."
4,Josh Hart,PTS,24.61,32.73,39.31,0.1640,0.4196,0.7066,4.04,13.73,27.77,"[0.2636203866432338, 0.4062116531968011, 0.444..."
5,Quentin Grimes,PTS,16.43,23.37,31.23,0.1716,0.4384,0.7327,2.82,10.25,22.88,"[0.3481012658227848, 0.5191059841384282, 0.403..."
6,Victor Wembanyama,PTS,22.42,30.90,37.56,0.4128,0.6497,0.9810,9.25,20.08,36.85,"[0.8184918529746116, 0.975609756097561, 0.7653..."
7,Anthony Edwards,PTS,16.42,23.52,32.21,0.3740,0.5987,0.9323,6.14,14.08,30.03,"[0.5377720870678617, 1.1001788908765653, 0.645..."
8,Julius Randle,PTS,26.30,35.10,40.67,0.3305,0.5506,0.8209,8.69,19.33,33.39,"[0.5172413793103449, 0.3089244851258581, 0.569..."
9,Stephon Castle,PTS,25.02,33.62,39.36,0.2929,0.5265,0.7731,7.33,17.70,30.43,"[0.1395889879798371, 0.830298616168973, 0.5798..."


In [9]:
from live import adjust_predictions

# Build contexts dict once (using the notebook's get_game_context)
game_contexts = {
    name: get_game_context(base_df, name, team_odds, is_playoff=True)
    for name in pts_preds["PLAYER_NAME"]
}

# Adjust the model's Q50 predictions with scenario signals
pts_preds = adjust_predictions(pts_preds, base_df, game_contexts)
ast_preds = adjust_predictions(ast_preds, base_df, game_contexts)
reb_preds = adjust_predictions(reb_preds, base_df, game_contexts)
pts_preds.head()

Jalen Brunson [PTS] [ix: home_favorite]  pace_bucket=low_pace  MIN: 37.1→37.4 (Δ+0.33)  RATE: 0.6415→0.6577 (Δ+0.0162)
Karl-Anthony Towns [PTS] [ix: home_favorite]  pace_bucket=low_pace  MIN: 31.5→31.3 (Δ-0.25)  RATE: 0.5196→0.4851 (Δ-0.0345)
OG Anunoby [PTS] [ix: home_favorite]  pace_bucket=low_pace  MIN: 35.6→35.8 (Δ+0.19)  RATE: 0.4698→0.4411 (Δ-0.0287)
Mikal Bridges [PTS] [ix: home_favorite]  pace_bucket=low_pace  MIN: 29.3→28.9 (Δ-0.39)  RATE: 0.3642→0.3000 (Δ-0.0643)
Josh Hart [PTS] [ix: home_favorite]  pace_bucket=low_pace  MIN: 32.1→31.5 (Δ-0.61)  RATE: 0.3950→0.3704 (Δ-0.0246)
Quentin Grimes [PTS] [ix: away_underdog]  pace_bucket=low_pace  MIN: 21.3→19.2 (Δ-2.08)  RATE: 0.3709→0.3034 (Δ-0.0676)
Victor Wembanyama [PTS] [ix: home_favorite]  pace_bucket=low_pace  MIN: 29.9→29.0 (Δ-0.96)  RATE: 0.7878→0.9259 (Δ+0.1381)
Anthony Edwards [PTS] [ix: away_underdog]  pace_bucket=low_pace  MIN: 23.7→23.9 (Δ+0.17)  RATE: 0.4904→0.3821 (Δ-0.1082)
Julius Randle [PTS] [ix: away_underdog]  pa

,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90,RATE_HISTORY,ADJ_CONTEXT_OK,ADJ_CONTEXT_ERR,ADJ_ACTIVE_STARS,ADJ_STARS_MISSING,ADJ_SPREAD_ROLE,ADJ_CTX_SPREAD,ADJ_CTX_TOTAL,ADJ_MIN_DELTA,ADJ_RATE_DELTA,ADJ_MIN_SHIFT,ADJ_RATE_SHIFT,ADJ_USED_INTERACTION,ADJ_MIN_LOG,ADJ_RATE_LOG
0,Jalen Brunson,PTS,29.87,37.40,42.23,0.4224,0.6577,0.9846,12.62,24.60,41.58,"[0.843687, 0.85963, 0.685666, 0.901572, 0.3564...",True,None,3,0,favorite,-7.5,212.5,0.3337,0.01620,0.33,0.0162,True,"{'stars': (0.0863, 133), 'pace': (0.1959, 53),...","{'stars': (0.0051, 133), 'pace': (-0.0255, 53)..."
1,Karl-Anthony Towns,PTS,23.96,31.25,36.51,0.2617,0.4851,0.8042,6.27,15.16,29.36,"[0.945964, 0.792185, 0.522626, 0.370711, 0.574...",True,None,3,0,favorite,-7.5,212.5,-0.2461,-0.03450,-0.25,-0.0345,True,"{'stars': (-0.0163, 133), 'pace': (0.8309, 54)...","{'stars': (-0.0208, 133), 'pace': (-0.03, 54),..."
2,OG Anunoby,PTS,27.95,35.76,41.20,0.1903,0.4411,0.7095,5.32,15.77,29.23,"[0.472457, 0.377383, 0.247478, 0.151933, 0.572...",True,None,3,0,favorite,-7.5,212.5,0.1908,-0.02870,0.19,-0.0287,True,"{'stars': (-0.0607, 128), 'pace': (0.8371, 50)...","{'stars': (-0.0294, 128), 'pace': (0.0059, 50)..."
3,Mikal Bridges,PTS,20.65,28.94,35.79,0.0339,0.3000,0.6010,0.70,8.68,21.51,"[0.309, 0.309691, 0.079082, 0.540583, 0.400523...",True,None,3,0,favorite,-7.5,212.5,-0.3918,-0.06425,-0.39,-0.0642,True,"{'stars': (-0.2129, 133), 'pace': (0.2949, 60)...","{'stars': (-0.035, 133), 'pace': (-0.0256, 60)..."
4,Josh Hart,PTS,23.39,31.51,38.09,0.1148,0.3704,0.6574,2.69,11.67,25.04,"[0.21442, 0.357012, 0.395025, 0.406407, 0.1514...",True,None,3,0,favorite,-7.5,212.5,-0.6090,-0.02460,-0.61,-0.0246,True,"{'stars': (-0.1842, 122), 'pace': (1.0297, 50)...","{'stars': (0.0125, 122), 'pace': (-0.0248, 50)..."


### Get Line Probabilities

In [10]:
all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, run_pts_simulation),
    line_probs_for_market(reb_preds, lines_dfs_reb, run_pts_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, run_pts_simulation),
], ignore_index=True)
all_line_probs.head()

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER
0,VJ Edgecombe,AST,3.5,35.93,43.65,46.94,0.31,3.02,6.59,0.306,0.694
1,Stephon Castle,AST,7.0,27.74,36.34,42.08,5.49,10.21,16.80,0.910,0.090
2,De'Aaron Fox,AST,6.0,23.25,31.81,37.50,2.79,6.54,12.05,0.659,0.341
3,Julius Randle,AST,4.5,26.56,35.36,40.93,0.66,3.65,6.91,0.297,0.703
4,Victor Wembanyama,AST,3.0,20.50,28.98,35.64,0.39,1.91,5.03,0.319,0.681


In [11]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='Underdog',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

_m = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left')
underdog_all_lines = _m.dropna(subset=[c for c in _m.columns if not str(c).startswith('ADJ_')])
underdog_all_lines.head()

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
5,Jaden McDaniels,AST,2.5,26.64,37.10,43.63,0.25,2.66,6.20,0.507,0.493,AST,Underdog,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,-105.0,-113.0,0.512,0.531,2.3,3.0,1.06,-0.2,0.5,0.189,0.425,0.575,-17.02,8.38,0.8,0.6,0.60,0.41,34.05,7.08,0.23,0.07,2.71,7.0
8,James Harden,AST,6.5,31.47,37.88,42.27,3.06,6.17,11.23,0.472,0.527,AST,Underdog,Toronto Raptors,-8.5,210.5,112.1,5.0,99.22,21.0,-106.0,-110.0,0.515,0.524,5.9,5.0,2.42,-0.6,-1.5,0.248,0.402,0.598,-21.88,14.16,0.4,0.4,0.47,0.73,35.19,4.93,0.27,0.06,6.70,10.0
10,LeBron James,AST,7.5,29.51,38.14,43.72,2.09,5.73,12.13,0.467,0.533,AST,Underdog,Oklahoma City Thunder,15.8,213.5,106.5,1.0,100.37,16.0,-108.0,-110.0,0.519,0.524,9.4,8.5,3.17,1.9,1.0,-0.599,0.725,0.275,39.63,-47.50,0.4,0.6,0.60,0.52,35.16,7.68,0.32,0.06,5.60,5.0
13,Ajay Mitchell,AST,3.5,23.68,33.18,38.39,1.21,4.36,8.17,0.635,0.365,AST,Underdog,Los Angeles Lakers,-15.8,213.5,115.5,20.0,99.22,22.0,-106.0,-111.0,0.515,0.526,3.7,3.0,1.95,0.2,-0.5,-0.103,0.541,0.459,5.14,-12.75,0.4,0.4,0.33,0.28,25.78,6.70,0.19,0.06,1.75,4.0
26,Rudy Gobert,REB,11.5,21.35,30.40,36.47,2.47,6.47,13.26,0.126,0.874,REB,Underdog,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,110.0,-116.0,0.476,0.537,10.9,11.0,3.28,-0.6,-0.5,0.183,0.427,0.573,-10.33,6.70,0.6,0.5,0.60,0.46,32.82,4.90,0.11,0.04,9.50,6.0


In [12]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='PrizePicks',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

_m = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left')
prizePicks_all_lines = _m.dropna(subset=[c for c in _m.columns if not str(c).startswith('ADJ_')])
prizePicks_all_lines.head(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
0,VJ Edgecombe,AST,3.5,35.93,43.65,46.94,0.31,3.02,6.59,0.306,0.694,AST,PrizePicks,New York Knicks,7.5,212.5,112.3,7.0,97.71,25.0,-114.0,135.0,0.533,0.426,4.5,3.5,2.99,1.0,0.0,-0.334,0.631,0.369,18.45,-13.28,0.4,0.5,0.47,0.56,37.95,3.75,0.19,0.05,3.50,4.0
1,Stephon Castle,AST,7.0,27.74,36.34,42.08,5.49,10.21,16.80,0.910,0.090,AST,PrizePicks,Minnesota Timberwolves,-14.0,216.5,112.5,8.0,101.50,10.0,-137.0,-137.0,0.578,0.578,7.7,7.5,2.83,0.7,0.5,-0.247,0.598,0.402,3.45,-30.46,0.2,0.5,0.60,0.27,33.03,3.97,0.25,0.04,4.00,6.0
2,De'Aaron Fox,AST,6.0,23.25,31.81,37.50,2.79,6.54,12.05,0.659,0.341,AST,PrizePicks,Minnesota Timberwolves,-14.0,216.5,112.5,8.0,101.50,10.0,-137.0,-137.0,0.578,0.578,6.4,6.5,2.17,0.4,0.5,-0.184,0.573,0.427,-0.88,-26.13,0.6,0.5,0.33,0.45,33.91,4.33,0.25,0.03,7.00,7.0
3,Julius Randle,AST,4.5,26.56,35.36,40.93,0.66,3.65,6.91,0.297,0.703,AST,PrizePicks,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,-137.0,-137.0,0.578,0.578,4.2,4.0,1.55,0.2,0.0,-0.129,0.551,0.449,-4.68,-22.33,0.6,0.4,0.33,0.47,33.35,2.97,0.27,0.04,5.14,7.0
5,Jaden McDaniels,AST,2.5,26.64,37.10,43.63,0.25,2.66,6.20,0.507,0.493,AST,PrizePicks,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,-105.0,-113.0,0.512,0.531,2.3,3.0,1.06,-0.2,0.5,0.189,0.425,0.575,-17.02,8.38,0.8,0.6,0.60,0.41,34.05,7.08,0.23,0.07,2.71,7.0


In [13]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='Betr DFS',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

_m = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left')
betr_all_lines = _m.dropna(subset=[c for c in _m.columns if not str(c).startswith('ADJ_')])
betr_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
93,James Harden,PTS,18.5,31.47,37.88,42.27,9.94,20.40,36.28,0.537,0.463,PTS,Betr DFS,Toronto Raptors,-8.5,210.5,112.1,5.0,99.22,21.0,-122.0,-103.0,0.550,0.507,21.3,20.5,4.08,2.8,2.0,-0.686,0.754,0.246,37.20,-51.52,0.4,0.7,0.53,0.71,35.19,4.93,0.27,0.06,22.40,10.0
111,Ajay Mitchell,PTS,15.5,23.68,33.18,38.39,4.29,14.08,26.24,0.366,0.634,PTS,Betr DFS,Los Angeles Lakers,-15.8,213.5,115.5,20.0,99.22,22.0,-115.0,-106.0,0.535,0.515,11.5,9.5,4.74,-4.0,-6.0,0.844,0.199,0.801,-62.80,55.67,0.2,0.1,0.13,0.22,25.78,6.70,0.19,0.06,9.25,4.0
39,Evan Mobley,REB,8.5,25.64,33.17,39.57,3.52,7.64,13.62,0.503,0.497,REB,Betr DFS,Toronto Raptors,-8.5,210.5,112.1,5.0,99.22,21.0,-113.0,-103.0,0.531,0.507,9.1,7.5,4.23,0.6,-1.0,-0.142,0.556,0.444,4.80,-12.49,0.6,0.4,0.47,0.56,31.69,5.46,0.22,0.05,9.14,14.0
28,Julius Randle,REB,6.5,26.56,35.36,40.93,1.65,5.43,11.24,0.430,0.570,REB,Betr DFS,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,110.0,-130.0,0.476,0.565,6.7,7.0,2.31,0.2,0.5,-0.087,0.535,0.465,12.35,-17.73,0.6,0.6,0.53,0.54,33.35,2.97,0.27,0.04,6.29,7.0
119,Cason Wallace,PTS,6.5,9.53,15.75,22.80,0.73,5.51,14.92,0.305,0.695,PTS,Betr DFS,Los Angeles Lakers,-15.8,213.5,115.5,20.0,99.22,22.0,-128.0,105.0,0.561,0.488,6.8,6.0,4.34,0.3,-0.5,-0.069,0.528,0.472,-5.95,-3.24,0.2,0.4,0.40,0.52,21.87,3.16,0.14,0.05,7.43,7.0


In [14]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='DraftKings Pick6',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

_m = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left')
draftKings_all_lines = _m.dropna(subset=[c for c in _m.columns if not str(c).startswith('ADJ_')])
draftKings_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
25,Quentin Grimes,REB,2.5,12.27,19.21,27.07,0.49,2.12,5.94,0.565,0.435,REB,DraftKings Pick6,New York Knicks,7.5,212.5,112.3,7.0,97.71,25.0,130.0,-115.0,0.435,0.535,3.2,3.0,1.23,0.7,0.5,-0.569,0.715,0.285,64.45,-46.72,0.6,0.7,0.73,0.69,22.74,4.31,0.17,0.07,3.67,6.0
26,Rudy Gobert,REB,11.5,21.35,30.40,36.47,2.47,6.47,13.26,0.126,0.874,REB,DraftKings Pick6,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,110.0,-116.0,0.476,0.537,10.9,11.0,3.28,-0.6,-0.5,0.183,0.427,0.573,-10.33,6.70,0.6,0.5,0.60,0.46,32.82,4.90,0.11,0.04,9.50,6.0
72,OG Anunoby,PTS,14.5,27.95,35.76,41.20,5.32,15.77,29.23,0.638,0.362,PTS,DraftKings Pick6,Philadelphia 76ers,-7.5,212.5,114.4,17.0,100.40,15.0,100.0,-130.0,0.500,0.565,19.7,20.0,8.90,5.2,5.5,-0.584,0.720,0.280,44.00,-50.46,0.8,0.7,0.67,0.64,33.30,7.34,0.18,0.05,18.29,7.0
3,Julius Randle,AST,4.5,26.56,35.36,40.93,0.66,3.65,6.91,0.297,0.703,AST,DraftKings Pick6,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,129.0,-135.0,0.437,0.574,4.2,4.0,1.55,-0.3,-0.5,0.194,0.423,0.577,-3.13,0.44,0.6,0.4,0.33,0.47,33.35,2.97,0.27,0.04,5.14,7.0
77,Anthony Edwards,PTS,20.5,16.76,23.86,32.55,2.64,9.12,23.30,0.103,0.897,PTS,DraftKings Pick6,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,115.0,-122.0,0.465,0.550,21.8,20.5,11.56,0.3,-1.0,-0.026,0.510,0.490,9.65,-10.84,0.6,0.5,0.53,0.72,30.48,7.93,0.31,0.04,28.00,7.0


In [15]:
all_line_probs = pd.concat([underdog_all_lines, prizePicks_all_lines, betr_all_lines, draftKings_all_lines])
all_line_probs.to_json('data/props/ev_analysis/all_line_probs.json', orient='records', lines=True)
all_line_probs.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
51,Isaiah Hartenstein,REB,8.5,9.32,16.88,23.18,1.59,5.15,10.54,0.259,0.741,REB,PrizePicks,Los Angeles Lakers,-15.8,213.5,115.5,20.0,99.22,22.0,-137.0,-137.0,0.578,0.578,8.8,8.0,3.97,0.8,0.0,-0.202,0.580,0.420,0.34,-27.34,0.4,0.4,0.47,0.56,21.14,4.87,0.14,0.03,9.57,7.0
41,Ausar Thompson,REB,7.0,24.75,32.87,39.53,3.14,7.07,12.91,0.660,0.340,REB,Betr DFS,Orlando Magic,-8.5,202.0,113.6,13.0,100.56,14.0,-137.0,-137.0,0.578,0.578,7.5,7.5,3.37,0.5,0.5,-0.148,0.559,0.441,-3.30,-23.71,0.8,0.5,0.40,0.23,30.03,5.87,0.14,0.04,7.92,13.0
44,Donovan Mitchell,REB,4.0,30.13,36.45,41.75,2.12,4.86,9.86,0.803,0.197,REB,PrizePicks,Toronto Raptors,-8.5,210.5,112.1,5.0,99.22,21.0,-137.0,-137.0,0.578,0.578,5.3,5.5,1.42,1.3,1.5,-0.915,0.820,0.180,41.85,-68.86,0.8,0.8,0.60,0.46,34.73,3.00,0.30,0.06,4.42,12.0
114,Marcus Smart,PTS,10.5,30.59,39.25,45.10,5.65,17.72,33.50,0.766,0.234,PTS,Underdog,Oklahoma City Thunder,15.8,213.5,106.5,1.0,100.37,16.0,105.0,-115.0,0.488,0.535,11.4,10.0,7.28,0.9,-0.5,-0.124,0.549,0.451,12.54,-15.68,0.6,0.5,0.47,0.39,31.33,6.19,0.18,0.06,14.00,2.0
29,Naz Reid,REB,6.5,17.35,24.38,30.98,1.52,4.71,10.71,0.379,0.621,REB,Betr DFS,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,130.0,-140.0,0.435,0.583,6.3,7.0,2.36,-0.2,0.5,0.085,0.466,0.534,7.18,-8.46,0.8,0.6,0.53,0.39,24.88,4.94,0.21,0.03,5.57,7.0


### Get top EVs for 2 legs

In [16]:
slate_path = build_greedy_slate(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks.json",
)
print(slate_path)

Legs: 87  |  Pairs: 885  |  Slate: 10  |  STRONG: 1  |  MARGINAL: 9  |  SKIP: 0  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\prizepicks.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\prizepicks.json


In [17]:
slate_path = build_greedy_slate(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog.json",
)
print(slate_path)

Legs: 26  |  Pairs: 105  |  Slate: 6  |  STRONG: 0  |  MARGINAL: 6  |  SKIP: 0  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\underdog.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\underdog.json


In [18]:
slate_path = build_greedy_slate(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings.json",
)
print(slate_path)

Legs: 33  |  Pairs: 90  |  Slate: 6  |  STRONG: 0  |  MARGINAL: 6  |  SKIP: 0  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\draftKings.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\draftKings.json


In [19]:
slate_path = build_greedy_slate(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr.json",
)
print(slate_path)

Legs: 77  |  Pairs: 655  |  Slate: 10  |  STRONG: 0  |  MARGINAL: 10  |  SKIP: 0  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\betr.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\betr.json


### Top EVs for 3 Legs

In [20]:
slate_path = build_greedy_slate_3leg(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks_3leg.json",
)
print(slate_path)

Legs: 87  |  Triples: 18373  |  Slate: 10  |  STRONG: 0  |  MARGINAL: 9  |  SKIP: 1  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\prizepicks_3leg.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\prizepicks_3leg.json


In [21]:
slate_path = build_greedy_slate_3leg(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog_3leg.json",
)
print(slate_path)

Legs: 26  |  Triples: 623  |  Slate: 5  |  STRONG: 0  |  MARGINAL: 5  |  SKIP: 0  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\underdog_3leg.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\underdog_3leg.json


In [22]:
slate_path = build_greedy_slate_3leg(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr_3leg.json",
)
print(slate_path)

Legs: 77  |  Triples: 12278  |  Slate: 9  |  STRONG: 0  |  MARGINAL: 9  |  SKIP: 0  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\betr_3leg.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\betr_3leg.json


In [23]:
slate_path = build_greedy_slate_3leg(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings_3leg.json",
)
print(slate_path)

Legs: 33  |  Triples: 502  |  Slate: 4  |  STRONG: 0  |  MARGINAL: 3  |  SKIP: 1  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\draftKings_3leg.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\draftKings_3leg.json
